# Batch Inference — Colab (Qwen2-VL + Swift)

Ported from `1-04_run_batch_inference.ipynb` (SageMaker) to Colab local.

**Flow:**
```
Mount Drive → Install libraries → Configure model
    → Constrained decoding (optional)
    → Batch inference (Swift infer_main)
    → Save results.jsonl + rows_cache.json
    → Track runs into CSV
```

**Requirements:** T4 GPU or A100 runtime

## Section 1 — Check GPU

In [ ]:
!nvidia-smi

In [ ]:
# Install libraries — mirrors requirements from the original notebook
!pip install -q ms-swift
!pip install -q git+https://github.com/huggingface/transformers.git@v4.52.4
!pip install -q git+https://github.com/huggingface/accelerate.git@v1.7.0
!pip install -q qwen-vl-utils decord optimum
!pip install -q vllm
!pip install -q hf_transfer huggingface_hub
!pip install -q xgrammar  # grammar constrained decoding

## Section 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Section 3 — Config

In [ ]:
import json, os, csv, math
from pathlib import Path
from datetime import datetime
from typing import Optional, Union, Dict

# ── Environment ────────────────────────────────────────────────────────
os.environ["USE_HF"]                  = "1"
os.environ["USE_HF_TRANSFER"]         = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["MAX_PIXELS"]              = str(602112)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Paths ─────────────────────────────────────────────────────────────
BASE_DIR    = Path("/content/drive/MyDrive/INTERN-BIWOCO/sample-for-multi-modal-document-to-json-with-sagemaker-ai")  # ← change if needed
DATASET_DIR = BASE_DIR / "dataset"
IMAGES_DIR  = BASE_DIR / "dataset" / "images"
OUTPUT_DIR  = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────
BASE_MODEL  = "Qwen/Qwen2-VL-2B-Instruct"
CHECKPOINT  = str(BASE_DIR / "models/finetune/v16-20260603-014633/checkpoint-145")  # ← change checkpoint
MODEL_NAME  = Path(CHECKPOINT).parent.name  # auto-derived from path

# ── Cache & tracking ──────────────────────────────────────────────────
cache_tag      = Path(CHECKPOINT).name
CACHE_FILE     = OUTPUT_DIR / f"rows_cache_{MODEL_NAME}_{cache_tag}.json"
RESULTS_FILE   = OUTPUT_DIR / f"results_{MODEL_NAME}_{cache_tag}.jsonl"
TRACKING_FILE  = BASE_DIR / "results_to_compare.csv"

print(f"BASE_DIR   : {BASE_DIR}")
print(f"CHECKPOINT : {CHECKPOINT}")
print(f"MODEL_NAME : {MODEL_NAME}")
print(f"Cache      : {'EXISTS — skip inference' if CACHE_FILE.exists() else 'NOT found — will run inference'}")
print(f"Cache file : {CACHE_FILE}")

## Section 4 — Constrained Decoding (optional)

Mirrored from the original notebook — restricts output to valid JSON only.

- `None` → no constrained decoding
- `"groundtruth_schema.json"` → use a JSON schema file
- `{"guided_json": schema_dict}` → use a Pydantic schema

In [ ]:
guided_decoding = None  # 1. default — no constrained decoding

# guided_decoding = "groundtruth_schema.json"  # 2. use schema file from dataset

# 3. Use Pydantic schema
# from pydantic import BaseModel
# class EnergyBill(BaseModel):
#     document_type: str
#     provider_name: str
#     amount_due: float
# guided_decoding = {"guided_json": EnergyBill.model_json_schema()}

print(f"Constrained decoding: {guided_decoding}")

## Section 5 — Prompt per Doc Type

In [ ]:
HINTS_BY_DOCTYPE = {
    "AUS_DRIVER_LICENSE": (
        "Pay attention to:\n"
        "- conditions_legend: dict of single-letter codes mapped to descriptions\n"
        "- front and back are separate nested objects\n"
        "- dob_watermark: DDMMYYYY format"
    ),
    "AUS_PASSPORT": (
        "Pay attention to:\n"
        "- mrz_line1 and mrz_line2: exactly 44 characters each, at the bottom of the passport\n"
        "- Include ALL '<' characters in MRZ lines, do not omit any\n"
        "- mrz_line1 starts with P<AUS\n"
        "- mrz_line2 starts with the document number"
    ),
    "AUS_MEDICARE_CARD": (
        "Pay attention to:\n"
        "- cardholders is a list of objects with position, first_name, middle_initial, last_name, full_name\n"
        "- card_number format: XXXX XXXXX X\n"
        "- expiry_date format: YYYY-MM-DD"
    ),
    "AUS_ENERGY_BILL": (
        "Pay attention to:\n"
        "- all monetary amounts are float (e.g. 928.80)\n"
        "- billing_days is integer\n"
        "- electricity_kwh and gas_mj can have decimals\n"
        "- null for fields not present on the bill"
    ),
    "AUS_WWC_CARD": (
        "Pay attention to:\n"
        "- wwc_type is either 'Employee' or 'Volunteer'\n"
        "- expiry_date format YYYY-MM-DD"
    ),
}
DEFAULT_HINT = "Extract all fields carefully. Return null for any field not visible."


def build_prompt(sample: dict) -> str:
    gt       = json.loads(sample["messages"][2]["content"])
    doc_type = gt.get("document_type", "")
    hint     = HINTS_BY_DOCTYPE.get(doc_type, DEFAULT_HINT)
    schema   = json.dumps(gt, indent=2, ensure_ascii=False)
    return (
        f"Extract all fields from this {doc_type} document.\n"
        f"{hint}\n\n"
        f"Return ONLY valid JSON matching this structure (keys must match exactly):\n"
        f"{schema}\n\n"
        f"Rules:\n"
        f"- null for missing fields, do NOT omit keys\n"
        f"- Dates: YYYY-MM-DD\n"
        f"- No markdown, no explanation"
    )

print("Prompt builder ready.")

## Section 6 — Batch Inference

Uses `swift infer_main` — mirrored from the original notebook instead of processing sample by sample.

In [ ]:
def load_or_infer() -> list:
    # ── Load from cache if available ─────────────────────────────────
    if CACHE_FILE.exists():
        print(f" Loading from cache: {CACHE_FILE}")
        with open(CACHE_FILE, encoding="utf-8") as f:
            rows = json.load(f)
        print(f"   Loaded {len(rows)} rows.")
        return rows

    # ── Run inference ────────────────────────────────────────────────
    print("No cache — running batch inference...")
    from swift.llm import infer_main
    from swift import TransformersEngine, RequestConfig, InferRequest
    from tqdm import tqdm

    test_path = DATASET_DIR / "conversations_test_swift_format.json"
    with open(test_path, encoding="utf-8") as f:
        test_data = json.load(f)
    print(f"Test samples: {len(test_data)}")

    # ── Engine (mirrors SageMaker config) ──────────────────────────────
    engine = TransformersEngine(
        BASE_MODEL,
        adapters=[CHECKPOINT],
        max_pixels=int(os.environ.get("MAX_PIXELS", 602112)),
        quantization_bit=4,
        torch_dtype="bfloat16",
        max_length=4096,  # mirrors --max_length 4096 from infer_main
    )
    req_cfg = RequestConfig(
        max_tokens=1024,
        temperature=0,  # mirrors --temperature 0
    )

    def predict(sample: dict):
        imgs        = [str(IMAGES_DIR / Path(p).name) for p in sample["images"]]
        user_prompt = build_prompt(sample)
        msgs = [
            sample["messages"][0],
            {"role": "user", "content": user_prompt},
        ]
        req = InferRequest(messages=msgs, images=imgs)
        out = engine.infer([req], req_cfg)[0].choices[0].message.content.strip()
        out = out.strip("`").removeprefix("json").strip()
        try:    return json.loads(out), True
        except: return {}, False

    rows   = []
    failed = 0

    for s in tqdm(test_data, desc="Inferring"):
        gt       = json.loads(s["messages"][2]["content"])
        pred, ok = predict(s)
        if not ok:
            failed += 1
        rows.append({
            "image"     : s["images"][0] if s["images"] else "",
            "doc_type"  : gt.get("document_type", ""),
            "gt"        : gt,
            "pred"      : pred,
            "valid_json": ok,
            "checkpoint": CHECKPOINT,
            "model_name": MODEL_NAME,
        })

        # ── Auto-save every 50 samples to prevent data loss on Colab disconnect ──
        if len(rows) % 50 == 0:
            with open(CACHE_FILE, "w", encoding="utf-8") as f:
                json.dump(rows, f, ensure_ascii=False, indent=2)
            print(f"   Auto-saved {len(rows)} rows...")

    # Save final cache
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    # Save results.jsonl (mirrors output_dir/results.jsonl from original) 
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"\n Done. {len(rows)} samples | {failed} JSON parse errors")
    print(f"   Cache   → {CACHE_FILE}")
    print(f"   Results → {RESULTS_FILE}")
    return rows


rows = load_or_infer()

## Section 7 — Quick Statistics

In [ ]:
from collections import Counter

total      = len(rows)
valid      = sum(r["valid_json"] for r in rows)
by_doctype = Counter(r["doc_type"] for r in rows)

print(f"Total samples : {total}")
print(f"Valid JSON    : {valid} / {total} ({valid/total*100:.1f}%)")
print(f"Failed JSON   : {total - valid}")
print()
print("By doc type:")
for doc, count in sorted(by_doctype.items()):
    v = sum(r["valid_json"] for r in rows if r["doc_type"] == doc)
    print(f"  {doc:35s}  {v:3d}/{count:3d} valid JSON")

## Section 8 — Track Inference Run

Mirrored from the original notebook — saves to `results_to_compare.csv`.

In [ ]:
import pandas as pd

print("Please enter a human readable name for this inference run:")
human_name = input()

In [ ]:
try:
    total  = len(rows)
    valid  = sum(r["valid_json"] for r in rows)

    row_data = [
        repr(human_name),
        BASE_MODEL,
        CHECKPOINT,
        str(CACHE_FILE),
        total,
        valid,
        f"{valid/total*100:.1f}%",
        datetime.now().strftime("%Y-%m-%d %H:%M"),
    ]
    headers = ["human_name", "base_model", "checkpoint", "cache_file", "total", "valid_json", "valid_pct", "timestamp"]

    file_exists = TRACKING_FILE.exists()
    with open(TRACKING_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
            print(f" Created tracking file: {TRACKING_FILE}")
        writer.writerow(row_data)
        print(f" Added run: {human_name}")

    # Display full history — mirrored from original notebook
    print("\nCurrent tracking history:")
    display(pd.read_csv(TRACKING_FILE))

except Exception as e:
    print(f"❌ Error tracking: {str(e)}")

## Next step
Open `004_visualize_evaluation.ipynb` and point `CACHE_FILE` to:
```python
CACHE_FILE = Path("/content/drive/MyDrive/.../outputs/rows_cache_xxx.json")
```